In [6]:
import pandas as pd
import altair as alt

dataset = pd.read_csv('michelin_my_maps.csv')
print(dataset.head())

                           Name  \
0  8 1/2 Otto e Mezzo - Bombana   
1                          ABaC   
2       AM par Alexandre Mazzia   
3                       Addison   
4                       Akelaŕe   

                                             Address  \
0  Shop 202, 2F, Alexandra House, 18 Chater Road,...   
1    Avenida del Tibidabo 1, Barcelona, 08022, Spain   
2     9 rue François-Rocca, Marseille, 13008, France   
3  Fairmont Grand Del Mar, 5200 Grand Del Mar Way...   
4  Paseo del Padre Orcolaga 56, Donostia / San Se...   

                          Location Price       Cuisine   Longitude   Latitude  \
0         Hong Kong, Hong Kong SAR  $$$$       Italian  114.158670  22.281464   
1                 Barcelona, Spain  $$$$      Creative    2.136511  41.410450   
2                Marseille, France  $$$$      Creative    5.386233  43.270110   
3               San Diego, CA, USA  $$$$  Contemporary -117.198891  32.941297   
4  Donostia / San Sebastián, Spain  $$$$      C

In [3]:
us = dataset[dataset["Location"].str.endswith("USA") & dataset["Award"].str.contains("Star")].copy()
us["Stars"] = us["Award"].str[0].astype(int)                     # "2 Stars" -> 2
us["City"] = us["Location"].str.rsplit(", ", n=1).str[0]          # "Chicago, IL"
 

In [4]:
cuisine_groups = {
    "Contemporary": "Contemporary / Creative", "Creative": "Contemporary / Creative",
    "Modern Cuisine": "Contemporary / Creative", "Fusion": "Contemporary / Creative",
    "American": "American", "American Contemporary": "American", "Californian": "American",
    "Barbecue": "American", "Steakhouse": "American", "Seafood": "American",
    "Japanese": "Japanese", "Sushi": "Japanese",
    "Korean": "Korean", "Korean Contemporary": "Korean",
    "Asian": "Other Asian", "Asian Contemporary": "Other Asian", "Chinese": "Other Asian",
    "Thai": "Other Asian", "Vietnamese": "Other Asian", "Filipino": "Other Asian",
    "Indian": "Other Asian",
    "French": "French", "French Contemporary": "French", "Modern French": "French",
    "Italian": "Other European", "Spanish": "Other European",
    "Spanish Contemporary": "Other European", "Scandinavian": "Other European",
    "Mexican": "Latin American", "Latin American": "Latin American",
    "Colombian": "Latin American", "Cuban": "Latin American", "Peruvian": "Latin American",
    "Middle Eastern": "Other", "Israeli": "Other", "Vegetarian": "Other",
}
us["Cuisine Group"] = us["Cuisine"].map(cuisine_groups).fillna("Other")

In [7]:
star_select = alt.selection_point(
    fields=["Stars"],
    bind=alt.binding_radio(options=[None, 1, 2, 3],
                           labels=["All", "1 ★", "2 ★", "3 ★"],
                           name="Michelin stars: "),
)

In [8]:
cuisine_select = alt.selection_point(fields=["Cuisine Group"], bind="legend")

In [10]:
states = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/us-atlas@3/states-10m.json", "states"
)
 
background = alt.Chart(states).mark_geoshape(
    fill="#eeeeee", stroke="white"
).project("albersUsa")

In [11]:
points = alt.Chart(us).mark_circle(stroke="white", strokeWidth=0.5).encode(
    longitude="Longitude:Q",                        # position channels = location
    latitude="Latitude:Q",
    color=alt.Color("Cuisine Group:N",              # color channel = cuisine
                    scale=alt.Scale(scheme="tableau10"),
                    legend=alt.Legend(title="Cuisine (click to filter)")),
    size=alt.Size("Stars:O", scale=alt.Scale(range=[40, 140, 300]),
                  legend=alt.Legend(title="Stars")),
    opacity=alt.condition(cuisine_select, alt.value(0.85), alt.value(0.05)),
    tooltip=["Name:N", "City:N", "Award:N", "Cuisine:N", "Price:N"],
).add_params(
    star_select, cuisine_select
).transform_filter(
    star_select
).project("albersUsa")

In [13]:
chart = (background + points).properties(
    width=900, height=550,
    title="Michelin-Starred Restaurants in the US",
)
chart

alt.LayerChart(...)